# Random Forest from Scratch — Simple Regression Example

**small Random Forest regressor from scratch**.

The core ideas are:

1. Train many decision trees.
2. Give each tree a different **bootstrap sample**.
3. At every split, consider only a **random subset of features**.
4. Average the predictions from all trees.

Unlike boosting methods, Random Forest trees are trained independently rather than sequentially.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Toy regression dataset

We use two input features so that random feature selection has a visible purpose.

In [ ]:
n_samples = 120

x1 = np.random.uniform(0, 10, n_samples)
x2 = np.random.uniform(-3, 3, n_samples)

X = np.column_stack([x1, x2])

y = (
    np.sin(x1)
    + 0.4 * x2
    + 0.15 * np.random.randn(n_samples)
)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(x1, y)
plt.xlabel("Feature 1")
plt.ylabel("Target")
plt.title("Toy regression dataset")
plt.show()

## 2. Random Forest prediction

If we train $T$ trees, the regression prediction is:

$$
\hat{y}(x)
=
\frac{1}{T}
\sum_{t=1}^{T}
f_t(x)
$$

Averaging many different trees reduces variance.

## 3. Bootstrap sampling

Each tree receives a dataset sampled **with replacement** from the original training set.

In [ ]:
def bootstrap_sample(X, y):
    n = len(X)

    indices = np.random.choice(
        n,
        size=n,
        replace=True
    )

    return X[indices], y[indices], indices

In [ ]:
X_boot, y_boot, indices = bootstrap_sample(X, y)

print("First 20 bootstrap indices:")
print(indices[:20])

print("\nUnique samples selected:")
print(len(np.unique(indices)), "out of", len(X))

## 4. Regression split criterion

For regression, we minimize squared error.

For a node:

$$
SSE = \sum_i (y_i-\bar{y})^2
$$

A candidate split is scored as:

$$
SSE_{split}
=
SSE_{left}
+
SSE_{right}
$$

The best split has the smallest total error.

In [ ]:
def squared_error(y):
    if len(y) == 0:
        return 0.0

    return np.sum(
        (y - y.mean()) ** 2
    )

## 5. Tree node

In [ ]:
class Node:
    def __init__(
        self,
        value=None,
        feature=None,
        threshold=None,
        left=None,
        right=None
    ):
        self.value = value
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right

    @property
    def is_leaf(self):
        return self.value is not None

## 6. Random feature selection

At each node, a Random Forest tree only considers a random subset of the available features.

This decorrelates the trees and makes averaging more effective.

In [ ]:
def choose_random_features(
    n_features,
    max_features
):
    return np.random.choice(
        n_features,
        size=max_features,
        replace=False
    )

## 7. Simple randomized regression tree

In [ ]:
class SimpleRegressionTree:

    def __init__(
        self,
        max_depth=5,
        min_samples_split=4,
        max_features=None
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.root = None

    def fit(self, X, y):

        if self.max_features is None:
            self.max_features = max(
                1,
                int(np.sqrt(X.shape[1]))
            )

        self.root = self._build_tree(
            X,
            y,
            depth=0
        )

        return self

    def _build_tree(
        self,
        X,
        y,
        depth
    ):
        if (
            depth >= self.max_depth
            or len(y) < self.min_samples_split
            or np.allclose(y, y[0])
        ):
            return Node(
                value=y.mean()
            )

        n_features = X.shape[1]

        feature_indices = choose_random_features(
            n_features,
            min(self.max_features, n_features)
        )

        best_feature = None
        best_threshold = None
        best_error = float("inf")

        for feature in feature_indices:

            values = np.unique(
                X[:, feature]
            )

            if len(values) <= 1:
                continue

            thresholds = (
                values[:-1] + values[1:]
            ) / 2

            for threshold in thresholds:

                left_mask = (
                    X[:, feature] < threshold
                )

                right_mask = ~left_mask

                if (
                    left_mask.sum() == 0
                    or right_mask.sum() == 0
                ):
                    continue

                error = (
                    squared_error(y[left_mask])
                    + squared_error(y[right_mask])
                )

                if error < best_error:
                    best_error = error
                    best_feature = feature
                    best_threshold = threshold

        if best_feature is None:
            return Node(
                value=y.mean()
            )

        left_mask = (
            X[:, best_feature]
            < best_threshold
        )

        right_mask = ~left_mask

        left_node = self._build_tree(
            X[left_mask],
            y[left_mask],
            depth + 1
        )

        right_node = self._build_tree(
            X[right_mask],
            y[right_mask],
            depth + 1
        )

        return Node(
            feature=best_feature,
            threshold=best_threshold,
            left=left_node,
            right=right_node
        )

    def _predict_one(self, x, node):

        if node.is_leaf:
            return node.value

        if x[node.feature] < node.threshold:
            return self._predict_one(
                x,
                node.left
            )

        return self._predict_one(
            x,
            node.right
        )

    def predict(self, X):
        return np.array([
            self._predict_one(x, self.root)
            for x in X
        ])

## 8. Random Forest regressor

Each tree is trained on a different bootstrap dataset.

The final prediction is the average of all tree predictions.

In [ ]:
class SimpleRandomForestRegressor:

    def __init__(
        self,
        n_estimators=30,
        max_depth=5,
        min_samples_split=4,
        max_features=None
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features

        self.trees = []

    def fit(self, X, y):

        self.trees = []

        for i in range(self.n_estimators):

            X_boot, y_boot, _ = bootstrap_sample(
                X,
                y
            )

            tree = SimpleRegressionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features
            )

            tree.fit(
                X_boot,
                y_boot
            )

            self.trees.append(tree)

            print(
                f"Trained tree {i + 1}/{self.n_estimators}"
            )

        return self

    def predict(self, X):

        all_predictions = np.array([
            tree.predict(X)
            for tree in self.trees
        ])

        return all_predictions.mean(
            axis=0
        )

## 9. Train the forest

In [ ]:
forest = SimpleRandomForestRegressor(
    n_estimators=40,
    max_depth=5,
    min_samples_split=5,
    max_features=1
)

forest.fit(X, y)

## 10. Training error

In [ ]:
predictions = forest.predict(X)

mse = np.mean(
    (y - predictions) ** 2
)

print("Training MSE:", mse)

## 11. Visualize predictions

We fix the second feature to zero and vary the first feature.

In [ ]:
x_test = np.linspace(
    0,
    10,
    300
)

X_test = np.column_stack([
    x_test,
    np.zeros_like(x_test)
])

y_pred = forest.predict(
    X_test
)

true_function = np.sin(
    x_test
)

plt.figure(figsize=(9, 4))

plt.plot(
    x_test,
    true_function,
    label="true function when x2 = 0"
)

plt.plot(
    x_test,
    y_pred,
    linewidth=2,
    label="Random Forest prediction"
)

plt.xlabel("Feature 1")
plt.ylabel("Target")
plt.title("Random Forest regression")
plt.legend()
plt.show()

## 12. Individual trees vs forest average

Individual trees can vary considerably, while their average is more stable.

In [ ]:
plt.figure(figsize=(9, 4))

for tree in forest.trees[:5]:

    plt.plot(
        x_test,
        tree.predict(X_test),
        alpha=0.45
    )

plt.plot(
    x_test,
    forest.predict(X_test),
    linewidth=3,
    label="forest average"
)

plt.xlabel("Feature 1")
plt.ylabel("Prediction")
plt.title("Individual trees and forest average")
plt.legend()
plt.show()

## 13. Effect of the number of trees

In [ ]:
tree_counts = [1, 3, 10, 20, 40]

for n in tree_counts:

    partial_predictions = np.array([
        tree.predict(X_test)
        for tree in forest.trees[:n]
    ]).mean(axis=0)

    plt.figure(figsize=(8, 3.5))

    plt.plot(
        x_test,
        true_function,
        label="true function"
    )

    plt.plot(
        x_test,
        partial_predictions,
        linewidth=2,
        label=f"{n} trees"
    )

    plt.xlabel("Feature 1")
    plt.ylabel("Prediction")
    plt.title(
        f"Random Forest with {n} trees"
    )
    plt.legend()
    plt.show()

## 14. Out-of-Bag intuition

Some training examples are not selected in a given bootstrap sample.

Those examples are called **out-of-bag samples** and can be used to estimate generalization performance.

In [ ]:
_, _, bootstrap_indices = bootstrap_sample(
    X,
    y
)

all_indices = np.arange(
    len(X)
)

oob_indices = np.setdiff1d(
    all_indices,
    np.unique(bootstrap_indices)
)

print("Training samples:", len(X))
print(
    "Unique bootstrap samples:",
    len(np.unique(bootstrap_indices))
)
print(
    "Out-of-bag samples:",
    len(oob_indices)
)

## 15. Random Forest vs Boosting

### Random Forest

Trees are trained independently:

```text
          data
      /    |    \
   tree1 tree2 tree3
      \\    |    /
        average
```

The goal is mainly to reduce variance.

### Boosting

Trees are trained sequentially:

```text
tree 1
   |
remaining error
   |
tree 2
   |
remaining error
   |
tree 3
```

Each new tree tries to improve the existing ensemble.

So:

$$
\text{Random Forest}
=
\text{independent randomized trees}
+
\text{averaging}
$$

while:

$$
\text{Boosting}
=
\text{sequential trees}
+
\text{error correction}
$$

## 16. Core equations and ideas

### Bootstrap sampling

$$
D_t \sim Bootstrap(D)
$$

Each tree sees a slightly different dataset.

### Random feature subsets

At each node, only some features are considered for splitting.

### Forest prediction

For regression:

$$
\hat y(x)
=
\frac{1}{T}
\sum_{t=1}^{T}
f_t(x)
$$

The combination of **bootstrap sampling**, **random feature selection**, and **averaging** is the central idea of Random Forest.

## 17. What production libraries add

Real implementations such as scikit-learn include:

- optimized tree construction,
- parallel training,
- out-of-bag scoring,
- feature importance,
- classification support,
- sample weights,
- many splitting criteria,
- memory and speed optimizations.

This notebook keeps only the essential ideas so the algorithm remains easy to inspect.